In [6]:
import sys
!{sys.executable} -m pip install tensorflow

import platform

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Architecture:", platform.architecture())
print("Platform:", platform.platform())

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: c:\Users\Jesse\Documents\Deepfake_Detection\.venv-tf\Scripts\python.exe
Architecture: ('64bit', 'WindowsPE')
Platform: Windows-11-10.0.26300-SP0


In [7]:
import sys
import platform

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("Architecture:", platform.architecture())
print("Platform:", platform.platform())

Python version: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Python executable: c:\Users\Jesse\Documents\Deepfake_Detection\.venv-tf\Scripts\python.exe
Architecture: ('64bit', 'WindowsPE')
Platform: Windows-11-10.0.26300-SP0


In [8]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, BatchNormalization
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
import cv2

In [9]:
def load_images(data_dir, img_size=(224, 224)):
    images = []
    labels = []
    
    for label in os.listdir(data_dir):
        label_dir = os.path.join(data_dir, label)
        if not os.path.isdir(label_dir):
            continue  # Skip non-directory files
        
        for img_file in os.listdir(label_dir):
            img_path = os.path.join(label_dir, img_file)

            # Read the image
            img = cv2.imread(img_path)

            if img is None:
                print(f"Warning: Could not read image {img_path}. Skipping...")
                continue  # Skip unreadable images

            # Resize and normalize the image
            img = cv2.resize(img, img_size)
            img = img.astype('float32') / 255.0  
            
            images.append(img)
            labels.append(label)

    return np.array(images), np.array(labels)  # Move return here

# Define dataset path
data_dir = r'C:\Users\Jesse\Documents\Deepfake_Detection\Dataset'

if os.path.exists(data_dir):
    print("Directory exists:", data_dir)
    print("Subfolders:", os.listdir(data_dir))
else:
    print("Error: Dataset directory does not exist!")

Directory exists: C:\Users\Jesse\Documents\Deepfake_Detection\Dataset
Subfolders: ['Test', 'Train', 'Validation']


In [10]:
for subset in ['Train', 'Test', 'Validation']:
    subset_path = os.path.join(data_dir, subset)
    
    if os.path.exists(subset_path):
        total_images = 0
        for label in os.listdir(subset_path):  # Iterate over "REAL" and "FAKE" folders
            label_dir = os.path.join(subset_path, label)
            if os.path.isdir(label_dir):  # Ensure it's a directory
                images = [f for f in os.listdir(label_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
                print(f"{subset}/{label} - Found {len(images)} images")
                total_images += len(images)

        print(f"Total images in {subset}: {total_images}\n")
    else:
        print(f"Error: {subset} folder not found!")


Train/Fake - Found 70001 images
Train/Real - Found 70001 images
Total images in Train: 140002

Test/Fake - Found 5492 images
Test/Real - Found 5413 images
Total images in Test: 10905

Validation/Fake - Found 19641 images
Validation/Real - Found 19787 images
Total images in Validation: 39428



In [11]:
broken_images = []

for subset in ['Train', 'Test', 'Validation']:
    subset_path = os.path.join(data_dir, subset)
    
    for label in os.listdir(subset_path):  # Iterate over "REAL" and "FAKE"
        label_dir = os.path.join(subset_path, label)
        
        for img_file in os.listdir(label_dir):
            img_path = os.path.join(label_dir, img_file)
            img = cv2.imread(img_path)
            
            if img is None: 
                broken_images.append(img_path)

print(f"Found {len(broken_images)} broken images")
if broken_images:
    print("Example broken images:", broken_images[:5])


Found 0 broken images


In [12]:
# Define paths to dataset folders
train_dir = r'C:\Users\Jesse\Documents\Deepfake_Detection\Dataset\Train'
val_dir = r'C:\Users\Jesse\Documents\Deepfake_Detection\Dataset\Validation'
test_dir = r'C:\Users\Jesse\Documents\Deepfake_Detection\Dataset\Test'

# Define parameters
IMG_SIZE = (128, 128)  # Resize images to 128x128
BATCH_SIZE = 16  # Reduce batch size to save memory

# Load dataset using TensorFlow pipeline
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    val_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

Found 140002 files belonging to 2 classes.
Found 39428 files belonging to 2 classes.
Found 10905 files belonging to 2 classes.


In [13]:
# Optimize data loading
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

print("Efficient data pipeline created. No MemoryErrors!")

# Define CNN model
def create_model():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')  # Binary classification: Real (0) or Fake (1)
    ])
    
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

Efficient data pipeline created. No MemoryErrors!


In [14]:
# Initialize model
model = create_model()
model.summary()  # Check model architecture

# Train the model
EPOCHS = 10  

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

c:\Users\Jesse\Documents\Deepfake_Detection\.venv-tf\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 126, 126, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 63, 63, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 61, 61, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 30, 30, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 28, 28, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     3,211,392 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,304,769 (12.61 MB)

 Trainable params: 3,304,769 (12.61 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10


c:\Users\Jesse\Documents\Deepfake_Detection\.venv-tf\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


8751/8751 ━━━━━━━━━━━━━━━━━━━━ 1093s 124ms/step - accuracy: 0.7454 - loss: 0.5394 - val_accuracy: 0.8352 - val_loss: 0.3670
Epoch 2/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 835s 95ms/step - accuracy: 0.8787 - loss: 0.2874 - val_accuracy: 0.8672 - val_loss: 0.3290
Epoch 3/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 839s 96ms/step - accuracy: 0.9030 - loss: 0.2434 - val_accuracy: 0.8841 - val_loss: 0.2823
Epoch 4/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 850s 97ms/step - accuracy: 0.9099 - loss: 0.2249 - val_accuracy: 0.8707 - val_loss: 0.3158
Epoch 5/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 848s 97ms/step - accuracy: 0.9073 - loss: 0.2337 - val_accuracy: 0.8829 - val_loss: 0.3209
Epoch 6/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 602s 69ms/step - accuracy: 0.9103 - loss: 0.2274 - val_accuracy: 0.8871 - val_loss: 0.2847
Epoch 7/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 729s 83ms/step - accuracy: 0.9117 - loss: 0.2257 - val_accuracy: 0.8770 - val_loss: 0.3073
Epoch 8/10
8751/8751 ━━━━━━━━━━━━━━━━━━━━ 834s 95ms/step - accuracy: 0.9188 - 

In [15]:
# Evaluate model
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

682/682 ━━━━━━━━━━━━━━━━━━━━ 15s 22ms/step - accuracy: 0.8727 - loss: 0.3999
Test Accuracy: 87.27%


In [16]:
model.save(r"C:\Users\Jesse\Documents\Deepfake_Detection\models\deepfake_detection_updated_v3.h5")